# 本次实验：官方权重串行推理，先跑最小合法 smoke

`ENABLE_TRANSFER = True` 只打开这份笔记本。库函数和 `kaggle_unet_load.ipynb` 默认仍然不传输。

顺序是文本编码器、退出并确认进程死亡、UNet 两卡采样、退出并确认 Ray worker 死亡、视频 VAE 解码、静音 MP4。三段权重不会同时驻留。音频 VAE 不下载；采样仍会处理音频 latent，成片没有音轨。

第一刀是 Comfy 节点允许的最小样本：32×32、5 帧、1 步、seed 1234。这是机制 smoke，不是训练长度的可用视频。官方 736×416×124 不在这次默认里，因为 T4 上的激活峰值还没有测量。

硬截止是 5340 秒，由独立监视进程在截止或可用内存低于 2GiB 时 SIGKILL。旧的 1080 秒 UNet 加载上限不用来杀掉这次推理。浏览器会话请留到约 100 分钟，命令返回后先下载 MP4 和 evidence，再 Stop。

本机没有 GPU 时不能把这次提交当成视频已生成。


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ENTRY_SCRIPT = Path("scripts") / "run_video_session.py"
MAX_DEPTH = 6


def default_seeds() -> list[Path]:
    return [Path.cwd(), Path("/kaggle/input"), Path("/kaggle/working")]


def _is_package(root: Path) -> bool:
    return (root / ENTRY_SCRIPT).is_file() and (root / "src" / "minimax_h3_t4").is_dir()


def _scan(seed: Path, max_depth: int) -> list[Path]:
    matches: list[Path] = []
    for current, dirnames, _filenames in os.walk(seed):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(seed).parts)
        except ValueError:
            depth = max_depth + 1
        if depth > max_depth:
            dirnames.clear()
            continue
        if _is_package(current_path):
            matches.append(current_path)
            dirnames.clear()
    return matches


def find_package_root() -> Path:
    pinned = os.environ.get("H3_T4_PACKAGE_ROOT", "").strip()
    if pinned:
        root = Path(pinned)
        if not _is_package(root):
            raise SystemExit(f"H3_T4_PACKAGE_ROOT is not the video package: {root}")
        return root.resolve()
    found: list[Path] = []
    seen: set[str] = set()
    for seed in default_seeds():
        if not seed.exists():
            continue
        for match in _scan(seed.resolve(), MAX_DEPTH):
            key = str(match.resolve())
            if key not in seen:
                seen.add(key)
                found.append(match.resolve())
    if not found:
        raise SystemExit("video package not found. Upload the folder that contains scripts/run_video_session.py")
    if len(found) > 1:
        raise SystemExit("more than one video package: " + ", ".join(str(path) for path in found))
    return found[0]


root = find_package_root()
evidence = Path("/kaggle/working/h3-video-smoke")
evidence.mkdir(parents=True, exist_ok=True)
ENABLE_TRANSFER = True
command = [
    sys.executable,
    str(root / "scripts" / "run_video_session.py"),
    "--evidence-dir",
    str(evidence),
    "--budget-s",
    "5340",
    "--mode",
    "smoke",
    "--comfy-dir",
    "/tmp/ComfyUI-h3",
]
if ENABLE_TRANSFER:
    command.append("--enable-transfer")
print("enable_transfer", ENABLE_TRANSFER)
print("mode", "smoke")
print("budget_s", 5340)
print("prompt_seed", 1234)
code = subprocess.call(command, cwd=str(root))
print("process_exit", code)
spec = importlib.util.spec_from_file_location("h3_video_session", root / "scripts" / "run_video_session.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module._show_video_files(evidence)
